In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="openai",
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model=os.environ.get("OPENAI_MODEL"),
    temperature=0.3,
)


agent = create_agent(
    model=model,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database from here. Which database are you referring to, and what is the table/collection name that stores artists (e.g., a table named artists or a collection named artists)?

If you want the total count, you can run one of these depending on your setup:

- SQL (MySQL, PostgreSQL, SQL Server, etc.)
  - SELECT COUNT(*) AS artist_count FROM artists;
  - If you want to exclude soft-deleted or inactive records: SELECT COUNT(*) AS artist_count FROM artists WHERE is_active = true; or WHERE deleted_at IS NULL;

- MongoDB (NoSQL)
  - db.artists.countDocuments({});

- If you need distinct counts (e.g., unique by an ID):
  - SQL: SELECT COUNT(DISTINCT id) FROM artists;
  - MongoDB: db.artists.distinct("id").length

If you share your database type and the relevant schema (table/collection name, and whether you track active vs deleted), I can give you the exact query and interpret the result. If you paste the command output, I’ll help you compute the count.
